In [1]:
import pandas as pd
df=pd.read_csv(r"C:\Users\HP\Documents\Solar_Power_Generation_Prediction\data\solar data.csv")

In [3]:
df.head()

,Date,Time,DateTime,Country,DC_Power,AC_Power,Ambient_Temperature,Module_Temperature,Irradiation,Day,Month,Hour,Daily_Yield,Total_Yield
0,2024-01-01,0,2024-01-01 00:00:00,Norway,0.0,0.0,-10.15,-3.33,0.039,1,1,0,145.722,145.722
1,2024-01-01,1,2024-01-01 01:00:00,Norway,0.0,0.0,-10.06,-3.50,0.014,1,1,1,145.722,291.444
2,2024-01-01,2,2024-01-01 02:00:00,Norway,0.0,0.0,-9.95,-5.17,0.025,1,1,2,145.722,437.166
3,2024-01-01,3,2024-01-01 03:00:00,Norway,0.0,0.0,-9.27,-4.53,0.000,1,1,3,145.722,582.888
4,2024-01-01,4,2024-01-01 04:00:00,Norway,0.0,0.0,-9.62,-6.34,0.000,1,1,4,145.722,728.610


In [4]:
df.dtypes

Date                    object
Time                     int64
DateTime                object
Country                 object
DC_Power               float64
AC_Power               float64
Ambient_Temperature    float64
Module_Temperature     float64
Irradiation            float64
Day                      int64
Month                    int64
Hour                     int64
Daily_Yield            float64
Total_Yield            float64
dtype: object

In [5]:
df["Date"]=pd.to_datetime(df["Date"])
df["DateTime"]=pd.to_datetime(df["DateTime"])

In [6]:
df.dtypes

Date                   datetime64[ns]
Time                            int64
DateTime               datetime64[ns]
Country                        object
DC_Power                      float64
AC_Power                      float64
Ambient_Temperature           float64
Module_Temperature            float64
Irradiation                   float64
Day                             int64
Month                           int64
Hour                            int64
Daily_Yield                   float64
Total_Yield                   float64
dtype: object

## Seperate X and y

In [16]:
X=df.drop(columns=["AC_Power","DC_Power","Date","Time","DateTime","Daily_Yield","Total_Yield"])
y=df["AC_Power"]

In [17]:
X.columns

Index(['Country', 'Ambient_Temperature', 'Module_Temperature', 'Irradiation',
       'Day', 'Month', 'Hour'],
      dtype='object')

## Train Test Split

In [19]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=42,test_size=0.2)

In [22]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(28108, 7)
(7028, 7)
(28108,)
(7028,)


## Missing values

In [28]:
print("Missing values in training data\n",X_train.isnull().sum().sum())
print("Missing values in testing data\n",X_test.isnull().sum().sum())

Missing values in training data
 0
Missing values in testing data
 0


## Numerical and Categorical columns

In [30]:
numerical_col=X_train.select_dtypes(include=["int64","float64"]).columns
numerical_col

Index(['Ambient_Temperature', 'Module_Temperature', 'Irradiation', 'Day',
       'Month', 'Hour'],
      dtype='object')

In [31]:
categorical_col=X_train.select_dtypes(include=["object","category"]).columns
categorical_col

Index(['Country'], dtype='object')

## Categorical Encoding

In [33]:
from sklearn.preprocessing import OneHotEncoder
encoder=OneHotEncoder(handle_unknown="ignore",sparse_output=False)
encoder

,categories,'auto'
,drop,None
,sparse_output,False
,dtype,<class 'numpy.float64'>
,handle_unknown,'ignore'
,min_frequency,None
,max_categories,None
,feature_name_combiner,'concat'


#### Applying encoding to training data

In [36]:
encoded_country=encoder.fit_transform(X_train[["Country"]])
encoded_country

array([[1., 0., 0., 0.],
       [1., 0., 0., 0.],
       [0., 0., 0., 1.],
       ...,
       [0., 0., 1., 0.],
       [1., 0., 0., 0.],
       [0., 0., 1., 0.]], shape=(28108, 4))

#### Checking encoded column names

In [37]:
encoder.get_feature_names_out(["Country"])

array(['Country_Australia', 'Country_Brazil', 'Country_India',
       'Country_Norway'], dtype=object)

#### Encoding testing data

In [39]:
encoding_test=encoder.transform(X_test[["Country"]])
encoding_test

array([[0., 0., 0., 1.],
       [0., 0., 1., 0.],
       [0., 1., 0., 0.],
       ...,
       [0., 0., 1., 0.],
       [0., 1., 0., 0.],
       [0., 1., 0., 0.]], shape=(7028, 4))

In [45]:
encoded_view=pd.DataFrame(encoded_country.astype(int),columns=encoder.get_feature_names_out(["Country"]))
encoded_view

,Country_Australia,Country_Brazil,Country_India,Country_Norway
0,1,0,0,0
1,1,0,0,0
2,0,0,0,1
3,0,1,0,0
4,0,0,1,0
...,...,...,...,...
28103,0,0,1,0
28104,0,1,0,0
28105,0,0,1,0
28106,1,0,0,0


## Feature Scaling

In [50]:
from sklearn.preprocessing import StandardScaler
numerical_features=X_train.select_dtypes(include=["int64","float64"]).columns
scaler=StandardScaler()

In [51]:
sca_train=scaler.fit_transform(X_train[numerical_features])
sca_train

array([[ 0.82168959,  0.73366931,  0.47163202, -0.76164533, -0.43815669,
        -0.36425873],
       [-0.57156218, -0.57608038,  0.48276608, -1.66870577,  1.30249746,
        -0.5085739 ],
       [-1.59626007, -1.54598494, -0.83940415, -0.87502789,  1.01238843,
        -0.79720425],
       ...,
       [ 1.23241212,  1.07267324, -0.7614657 ,  1.27924065, -0.72826572,
        -1.0858346 ],
       [-0.95192329, -0.77460378, -0.77259976, -0.76164533, -1.59859279,
         1.22320818],
       [ 1.55711063,  1.61945378, -0.83940415, -0.30811512, -0.14804767,
        -1.23014977]], shape=(28108, 6))

In [53]:
sca_test=scaler.transform(X_test[numerical_features])
sca_test

array([[-2.40252441, -2.32325451,  1.98308136,  1.05247554,  1.30249746,
         0.06868679],
       [ 1.48879743,  1.48317925,  0.50503421, -0.53488022,  0.14206136,
        -0.5085739 ],
       [ 0.37723216,  0.22390161, -0.81991954,  1.50600576, -1.30848377,
         1.6561537 ],
       ...,
       [ 0.25156962,  0.18688877, -0.83940415,  0.82571043,  1.01238843,
        -0.94151942],
       [ 0.09638903,  0.13221072, -0.0433185 ,  0.59894532,  1.30249746,
         0.79026266],
       [ 0.17060583,  0.17258836,  0.27400236, -1.101793  ,  1.30249746,
        -0.5085739 ]], shape=(7028, 6))

In [54]:
from sklearn.compose import ColumnTransformer
preprocess=ColumnTransformer(transformers=[("number",StandardScaler(), numerical_features),("category", OneHotEncoder(handle_unknown="ignore"), ["Country"])])
preprocess

,transformers,"[('number', ...), ('category', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,copy,True
,with_mean,True
,with_std,True


## Baseline model

In [57]:
baseline_prediction=[y_train.mean()]*len(y_test)
baseline_prediction

[np.float64(16.48910986196101),
 np.float64(16.48910986196101),
 np.float64(16.48910986196101),
 np.float64(16.48910986196101),
 np.float64(16.48910986196101),
 np.float64(16.48910986196101),
 np.float64(16.48910986196101),
 np.float64(16.48910986196101),
 np.float64(16.48910986196101),
 np.float64(16.48910986196101),
 np.float64(16.48910986196101),
 np.float64(16.48910986196101),
 np.float64(16.48910986196101),
 np.float64(16.48910986196101),
 np.float64(16.48910986196101),
 np.float64(16.48910986196101),
 np.float64(16.48910986196101),
 np.float64(16.48910986196101),
 np.float64(16.48910986196101),
 np.float64(16.48910986196101),
 np.float64(16.48910986196101),
 np.float64(16.48910986196101),
 np.float64(16.48910986196101),
 np.float64(16.48910986196101),
 np.float64(16.48910986196101),
 np.float64(16.48910986196101),
 np.float64(16.48910986196101),
 np.float64(16.48910986196101),
 np.float64(16.48910986196101),
 np.float64(16.48910986196101),
 np.float64(16.48910986196101),
 np.floa

##### The baseline is a reference point.Our regression models should perform better than this simple average prediction.

In [83]:
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
import numpy as np
print("MAE: ",mean_absolute_error(y_test,baseline_prediction))
print("MSE: ",mean_squared_error(y_test,baseline_prediction))
print("RMSE: ",np.sqrt(mean_squared_error(y_test,baseline_prediction)))
print("R2 Score: ",r2_score(y_test,baseline_prediction))

MAE:  17.951311781619783
MSE:  426.33402447811477
RMSE:  20.647857624415053
R2 Score:  -0.0006891308029446375


## Feature Engineering

#### Here we created a new feature named "Temperature_Difference" as it is the difference between the other two features module temperature and ambient temperature. 

In [62]:
X_train["Temperature_Difference"]=(X_train["Module_Temperature"]-X_train["Ambient_Temperature"])
X_test["Temperature_Difference"]=(X_test["Module_Temperature"]-X_test["Ambient_Temperature"])

In [65]:
X_train.head()

,Country,Ambient_Temperature,Module_Temperature,Irradiation,Day,Month,Hour,Temperature_Difference
12465,Australia,29.22,33.20,0.471,9,5,9,3.98
29360,Australia,12.70,17.63,0.475,1,11,8,4.93
26982,Norway,0.55,6.10,0.000,8,10,6,5.55
7725,Brazil,25.11,31.52,0.006,21,3,21,6.41
13592,India,36.00,40.42,0.582,21,5,8,4.42


In [66]:
X_test.head()

,Country,Ambient_Temperature,Module_Temperature,Irradiation,Day,Month,Hour,Temperature_Difference
31596,Norway,-9.01,-3.14,1.014,25,11,12,5.87
18488,India,37.13,42.11,0.483,11,7,8,4.98
5711,Brazil,23.95,27.14,0.007,29,2,23,3.19
10454,Australia,24.84,31.74,0.750,18,4,14,6.90
31171,India,19.45,24.26,0.033,20,11,19,4.81


In [69]:
print(X_train.shape)
print(X_test.shape)

(28108, 8)
(7028, 8)


In [71]:
numerical_features

Index(['Ambient_Temperature', 'Module_Temperature', 'Irradiation', 'Day',
       'Month', 'Hour', 'Temperature_Difference'],
      dtype='object')

#### So now we have 7 numerical features and also we neded to update it on the ColumnTransformer

In [73]:
from sklearn.compose import ColumnTransformer
preprocess=ColumnTransformer(transformers=[("number",StandardScaler(), numerical_features),("category", OneHotEncoder(handle_unknown="ignore"), ["Country"])])
preprocess

,transformers,"[('number', ...), ('category', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,copy,True
,with_mean,True
,with_std,True


## Model Building

### Linear Regression

In [76]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

In [78]:
linear_model=Pipeline([("preprocessor", preprocess),("regressor", LinearRegression())])
linear_model

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('number', ...), ('category', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [80]:
linear_model.fit(X_train,y_train)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('number', ...), ('category', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [81]:
y_pred_linear=linear_model.predict(X_test)
y_pred_linear

array([54.04521351, 25.83127046,  2.81545195, ..., -0.31647587,
       18.50835827, 24.79085174], shape=(7028,))

In [84]:
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
import numpy as np
print("MAE: ",mean_absolute_error(y_test,y_pred_linear))
print("MSE: ",mean_squared_error(y_test,y_pred_linear))
print("RMSE: ",np.sqrt(mean_squared_error(y_test,y_pred_linear)))
print("R2 Score: ",r2_score(y_test,y_pred_linear))

MAE:  2.0695412212487176
MSE:  7.725707328970158
RMSE:  2.7795156644584966
R2 Score:  0.9818662576571766


##### Linear Regression performed  better than the baseline model.

### Decision Tree Regressor

In [85]:
from sklearn.tree import DecisionTreeRegressor

In [92]:
decision_tree_model=Pipeline([("preprocessor", preprocess),("regressor",DecisionTreeRegressor(random_state=42))])
decision_tree_model

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('number', ...), ('category', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [93]:
decision_tree_model.fit(X_train,y_train)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('number', ...), ('category', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [94]:
y_pred_tree=decision_tree_model.predict(X_test)
y_pred_tree

array([48.678, 25.906,  0.   , ...,  0.   , 19.339, 25.711], shape=(7028,))

In [95]:
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
import numpy as np
print("MAE: ",mean_absolute_error(y_test,y_pred_tree))
print("MSE: ",mean_squared_error(y_test,y_pred_tree))
print("RMSE: ",np.sqrt(mean_squared_error(y_test,y_pred_tree)))
print("R2 Score: ",r2_score(y_test,y_pred_tree))

MAE:  0.8139214570290267
MSE:  2.280999223676722
RMSE:  1.510297726832932
R2 Score:  0.994646049812005


#### Decision Tree performed better than Linear Regression on the test dataset.

### Random Forest Regressor

In [97]:
from sklearn.ensemble import RandomForestRegressor

In [98]:
random_forest_model=Pipeline([("preprocessor", preprocess),("regressor",RandomForestRegressor(random_state=42))])
random_forest_model

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('number', ...), ('category', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [100]:
random_forest_model.fit(X_train,y_train)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('number', ...), ('category', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [101]:
y_pred_forest=random_forest_model.predict(X_test)
y_pred_forest

array([45.45759, 26.28236,  0.     , ...,  0.     , 18.48817, 26.61673],
      shape=(7028,))

In [102]:
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
import numpy as np
print("MAE: ",mean_absolute_error(y_test,y_pred_forest))
print("MSE: ",mean_squared_error(y_test,y_pred_forest))
print("RMSE: ",np.sqrt(mean_squared_error(y_test,y_pred_forest)))
print("R2 Score: ",r2_score(y_test,y_pred_forest))

MAE:  0.5977137734775183
MSE:  1.2524261114633466
RMSE:  1.1191184528294342
R2 Score:  0.9970603115751567


#### Random Forest performed better than Linear Regression and Decision Tree on the test dataset.

### Gradient Boosting Regressor

In [104]:
from sklearn.ensemble import GradientBoostingRegressor

In [107]:
gradient_boosting_model=Pipeline([("preprocessor", preprocess),("regressor",GradientBoostingRegressor(random_state=42))])
gradient_boosting_model

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('number', ...), ('category', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [108]:
gradient_boosting_model.fit(X_train,y_train)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('number', ...), ('category', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [114]:
y_pred_gradboost=gradient_boosting_model.predict(X_test)
y_pred_gradboost

array([ 4.57122777e+01,  2.64407904e+01,  4.08978456e-02, ...,
       -2.69850275e-02,  1.80073250e+01,  2.69240412e+01], shape=(7028,))

In [115]:
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
import numpy as np
print("MAE: ",mean_absolute_error(y_test,y_pred_gradboost))
print("MSE: ",mean_squared_error(y_test,y_pred_gradboost))
print("RMSE: ",np.sqrt(mean_squared_error(y_test,y_pred_gradboost)))
print("R2 Score: ",r2_score(y_test,y_pred_gradboost))

MAE:  0.6685942360587077
MSE:  1.3376727484368804
RMSE:  1.1565780338727174
R2 Score:  0.9968602210870439


#### So here Gradient Boosting performed better than Linear Regression and Decision Tree but slightly worse than Random Forest on the test dataset.

### Voting Regressor

In [116]:
from sklearn.ensemble import VotingRegressor

In [118]:
voting_model=VotingRegressor(estimators=[
("Linear Regression",linear_model),
("Decision tree",decision_tree__model),
("Random Forest",random_forest_model),
("Gradient Boosting",gradient_boosting_model)   
]
)
voting_model

,estimators,"[('Linear Regression', ...), ('Decision tree', ...), ...]"
,weights,None
,n_jobs,None
,verbose,False
,transformers,"[('number', ...), ('category', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [119]:
voting_model.fit(X_train,y_train)

,estimators,"[('Linear Regression', ...), ('Decision tree', ...), ...]"
,weights,None
,n_jobs,None
,verbose,False
,transformers,"[('number', ...), ('category', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [121]:
y_pred_voting=voting_model.predict(X_test)
y_pred_voting

array([48.4732703 , 26.11510523,  0.71408745, ..., -0.08586522,
       18.58571331, 26.01065574], shape=(7028,))

In [122]:
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
import numpy as np
print("MAE: ",mean_absolute_error(y_test,y_pred_voting))
print("MSE: ",mean_squared_error(y_test,y_pred_voting))
print("RMSE: ",np.sqrt(mean_squared_error(y_test,y_pred_voting)))
print("R2 Score: ",r2_score(y_test,y_pred_voting))

MAE:  0.8792172686954706
MSE:  1.6686442429219448
RMSE:  1.2917601336633457
R2 Score:  0.996083366418824


#### Combining multiple algorithms did not improve the prediction performance in this experiment.

#### Model,MAE,RMSE,R²
#### Linear Regression	2.070,2.780,0.9819
#### Decision Tree	0.814,1.510,0.9946
#### Random Forest	0.598,1.119,0.9971
#### Gradient Boosting	0.669,1.157,0.9969
#### Voting Regressor	0.879,1.292,0.9961

#### So best model is Random Forest

## Cross Validation

In [123]:
from sklearn.model_selection import cross_val_score

#### Cross Validation for Linear Regression

In [124]:
cv_linear=cross_val_score(linear_model,X_train,y_train,cv=5,scoring="r2")
cv_linear

array([0.98146901, 0.98140345, 0.98172048, 0.98135906, 0.98211125])

In [125]:
cv_linear.mean()

np.float64(0.9816126520058062)

#### Cross Validation for Decision Tree

In [126]:
cv_decision_tree=cross_val_score(decision_tree__model,X_train,y_train,cv=5,scoring="r2")
cv_decision_tree

array([0.99440627, 0.99482976, 0.99436558, 0.99464643, 0.99449267])

In [127]:
cv_decision_tree.mean()

np.float64(0.9945481437415653)

#### Cross Validation for Random Forest

In [131]:
cv_random_forest=cross_val_score(random_forest_model,X_train,y_train,cv=5,scoring="r2")
cv_random_forest

array([0.99713303, 0.99706014, 0.99706061, 0.99705831, 0.99704828])

In [132]:
cv_random_forest.mean()

np.float64(0.997072071955745)

#### Cross Validation for Gradient Boosting 

In [133]:
cv_gradient_boosting=cross_val_score(gradient_boosting_model,X_train,y_train,cv=5,scoring="r2")
cv_gradient_boosting

array([0.99699933, 0.99696695, 0.99693198, 0.99694573, 0.99687965])

In [134]:
cv_gradient_boosting.mean()

np.float64(0.9969447293929189)